# Single reconstruction → black-on-white density PNG

Loads **one STRND reconstruction CSV**, allows you to rotate and flip it, converts its points into a **sliding-window density image**, and exports a **publication-quality, borderless PNG**.

Steps: `normalize_pos_95` (center, scale so the central 95% lies in [-1, 1]), optional rotation/flip, rasterization, integral-image sliding-window counts, and **P10/P90 contrast mapping**. Only the reconstruction CSV from pipeline step 07 is needed; no graph file.

**Contrast:** densities at or below the **10th percentile (0.1 quantile)** are white (255); at or above the **90th percentile (0.9 quantile)** are black (0); intermediate values are linearly mapped to grays. Percentiles are calculated **separately for this image**.

**Run in order:** (1) imports → (2) settings → (3) helper functions → (4) load + orient → (5) preview density → (6) save. Edit the settings and rerun sections 4–6 to adjust the image. The PNG contains the image only, with no axes, title, or extra margins.

## 1. Imports

In [ ]:
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


## 2. Settings — edit here

- Set `INPUT_CSV` to any CSV with **`x` and `y`** columns (optional `node_ID` and `in_lcc` are supported).
- `ROTATION_DEG` is **counterclockwise**, in degrees; flips are applied **before** rotation.
- `WINDOW_WIDTH_NORM` is the **full width of the square window**, in coordinates after `normalize_pos_95`.
- With the 4096 × 4096 grid and the default window size, `STEP_FRACTION` is the stride as a fraction of the window width: the default 0.5 (half a window) gives an 8 px stride and a 512 × 512 image, while 0.01 gives a 1 px stride (a full-resolution 4096 × 4096 image). A smaller grid, e.g. `(2048, 2048)`, is faster.
- `XLIM`/`YLIM` define the exported frame. The default `(-1.2, 1.2)` matches the frame used in the `evaluate/` notebooks.


In [ ]:
# ----------- REQUIRED: reconstruction CSV -----------
DATA_DIR = Path(os.environ.get("SPAN_DATA_ROOT", "./orcd")) / "scratch" / "260921Jac"
INPUT_CSV = DATA_DIR / Path("M1_positions_postPipeline0302_Lev3_min_e4_min_d2_max_d30_LCC_STRND_d32_rnd123.csv")  # <-- EDIT THIS

# ----------- Orientation -----------
ROTATION_DEG = 0.0       # counterclockwise (can be negative / non-integer)
FLIP_HORIZONTAL = False # mirror left-to-right before rotation
FLIP_VERTICAL = True   # mirror top-to-bottom before rotation
X_SHIFT = 0.0           # optional pan in normalized coordinates
Y_SHIFT = 0.0
EXTRA_SCALE = 1.0       # >1 zooms in; <1 zooms out

# ----------- Sliding-window density -----------
WINDOW_WIDTH_NORM = 0.01
IMAGE_SHAPE = (4096, 4096)  # (height, width), high-resolution output
XLIM = (-1.2, 1.2)
YLIM = (-1.2, 1.2)
STEP_FRACTION = 0.5     # stride as a fraction of the window width (0.5 = half a window); 0.01 gives a 1 px stride

# ----------- Black-on-white percentile normalization -----------
LOW_PERCENTILE = 10.0    # 0.1 quantile -> white
HIGH_PERCENTILE = 90.0   # 0.9 quantile -> black
PERCENTILE_POPULATION = "all"  # "all" or "nonzero" (only positive windows)

# ----------- Output / preview -----------
OUTPUT_DIR = None        # None -> INPUT_CSV.parent / "density_exports"
OUTPUT_FILENAME = None   # None -> automatically named from INPUT_CSV
PNG_DPI = 600           # metadata; image resolution is determined by IMAGE_SHAPE + stride
SAVE_SETTINGS_JSON = True
PREVIEW_SCATTER_POINTS = 30000   # sampled points in scatter preview, for speed
SCATTER_SEED = 12345


## 3. Processing helpers

In [ ]:
def normalize_pos_95(pos):
    pos = np.asarray(pos, dtype=np.float64)
    if pos.ndim != 2 or pos.shape[1] != 2:
        raise ValueError(f"pos must be Nx2; got {pos.shape}")

    finite = np.isfinite(pos).all(axis=1)
    pf = pos[finite]
    if pf.shape[0] == 0:
        raise ValueError("No finite positions to normalize.")

    mu = pf.mean(axis=0)
    p0 = pf - mu

    qlo = np.percentile(p0, 2.5, axis=0)
    qhi = np.percentile(p0, 97.5, axis=0)
    s = np.maximum(np.abs(qlo), np.abs(qhi)).max()

    if not np.isfinite(s) or s == 0:
        raise ValueError("Degenerate layout: cannot determine nonzero scale.")

    pos_norm = np.full_like(pos, np.nan)
    pos_norm[finite] = p0 / s
    return pos_norm, finite



def apply_anchor_transform(
    pos_norm,
    rotation_deg=0.0,
    flip_h=False,
    flip_v=False,
    x_shift=0.0,
    y_shift=0.0,
    extra_scale=1.0,
):
    pos = np.asarray(pos_norm, dtype=np.float64).copy()

    sx = -1.0 if bool(flip_h) else 1.0
    sy = -1.0 if bool(flip_v) else 1.0
    pos[:, 0] *= sx
    pos[:, 1] *= sy

    theta = np.deg2rad(float(rotation_deg))
    c = np.cos(theta)
    s = np.sin(theta)
    R = np.array([[c, -s], [s, c]], dtype=np.float64)
    pos = pos @ R.T

    pos *= float(extra_scale)
    pos[:, 0] += float(x_shift)
    pos[:, 1] += float(y_shift)
    return pos



def rasterize_point_counts(
    pos_norm,
    image_shape=(2048, 2048),
    xlim=(-1.2, 1.2),
    ylim=(-1.2, 1.2),
):
    """
    Rasterize normalized XY coordinates into an integer point-count image.

    Multiple points that land in the same raster pixel are retained as
    multiple counts.

    Image orientation:
        left   = xmin
        right  = xmax
        top    = ymax
        bottom = ymin
    """
    pos_norm = np.asarray(pos_norm, dtype=np.float64)

    H, W = map(int, image_shape)
    xmin, xmax = map(float, xlim)
    ymin, ymax = map(float, ylim)

    finite = np.isfinite(pos_norm).all(axis=1)
    p = pos_norm[finite]

    in_bounds = (
        (p[:, 0] >= xmin) & (p[:, 0] <= xmax) &
        (p[:, 1] >= ymin) & (p[:, 1] <= ymax)
    )

    p = p[in_bounds]

    if len(p) == 0:
        raise ValueError(
            "No reconstruction points fall inside the selected image bounds."
        )

    cols = np.floor(
        (p[:, 0] - xmin) / (xmax - xmin) * W
    ).astype(np.int64)

    rows = np.floor(
        (ymax - p[:, 1]) / (ymax - ymin) * H
    ).astype(np.int64)

    cols = np.clip(cols, 0, W - 1)
    rows = np.clip(rows, 0, H - 1)

    point_image = np.zeros((H, W), dtype=np.uint32)
    np.add.at(point_image, (rows, cols), 1)

    dx = (xmax - xmin) / W
    dy = (ymax - ymin) / H

    info = {
        "height_px": H,
        "width_px": W,
        "xlim": [xmin, xmax],
        "ylim": [ymin, ymax],
        "dx_normalized_per_pixel": float(dx),
        "dy_normalized_per_pixel": float(dy),
        "finite_points_total": int(finite.sum()),
        "points_in_bounds": int(len(p)),
        "points_outside_bounds": int(finite.sum() - len(p)),
        "occupied_raster_pixels": int(np.count_nonzero(point_image)),
        "max_points_in_one_raster_pixel": int(point_image.max()),
    }

    return point_image, info



def box_window_counts(
    point_image,
    window_height_px,
    window_width_px,
    step_y_px=1,
    step_x_px=1,
):
    """
    Exact rectangular sliding-window sum using an integral image,
    evaluated only on a strided grid of window centers.

    Returns a lower-resolution count image if step_x_px or step_y_px > 1.
    """
    H, W = point_image.shape

    window_height_px = max(1, int(window_height_px))
    window_width_px = max(1, int(window_width_px))
    step_y_px = max(1, int(step_y_px))
    step_x_px = max(1, int(step_x_px))

    if window_height_px % 2 == 0:
        window_height_px += 1
    if window_width_px % 2 == 0:
        window_width_px += 1

    ry = window_height_px // 2
    rx = window_width_px // 2

    sat = np.zeros((H + 1, W + 1), dtype=np.uint64)
    sat[1:, 1:] = point_image
    np.cumsum(sat, axis=0, out=sat)
    np.cumsum(sat, axis=1, out=sat)

    row_centers = np.arange(0, H, step_y_px)
    col_centers = np.arange(0, W, step_x_px)

    r0 = np.maximum(row_centers - ry, 0)
    r1 = np.minimum(row_centers + ry + 1, H)
    c0 = np.maximum(col_centers - rx, 0)
    c1 = np.minimum(col_centers + rx + 1, W)

    counts = (
        sat[r1[:, None], c1[None, :]]
        - sat[r0[:, None], c1[None, :]]
        - sat[r1[:, None], c0[None, :]]
        + sat[r0[:, None], c0[None, :]]
    )

    return counts



def local_density_map(
    pos_norm,
    image_shape=(2048, 2048),
    window_width_norm=0.04,
    step_fraction=0.5,
    xlim=(-1.2, 1.2),
    ylim=(-1.2, 1.2),
):
    """
    Create the exact sliding-window count map.

    window_width_norm is the full width of the square counting window in
    normalized reconstruction coordinates.
    """
    point_image, info = rasterize_point_counts(
        pos_norm=pos_norm,
        image_shape=image_shape,
        xlim=xlim,
        ylim=ylim,
    )

    dx = info["dx_normalized_per_pixel"]
    dy = info["dy_normalized_per_pixel"]

    wx = max(1, int(round(window_width_norm / dx)))
    wy = max(1, int(round(window_width_norm / dy)))

    if wx % 2 == 0:
        wx += 1
    if wy % 2 == 0:
        wy += 1

    step_x = max(1, int(round(wx * step_fraction)))
    step_y = max(1, int(round(wy * step_fraction)))

    counts = box_window_counts(
        point_image,
        window_height_px=wy,
        window_width_px=wx,
        step_y_px=step_y,
        step_x_px=step_x,
    )

    info.update({
        "requested_window_width_normalized": float(window_width_norm),
        "window_width_px": int(wx),
        "window_height_px": int(wy),
        "actual_window_width_normalized": float(wx * dx),
        "actual_window_height_normalized": float(wy * dy),
        "step_fraction": float(step_fraction),
        "step_x_px": int(step_x),
        "step_y_px": int(step_y),
        "max_window_count": int(counts.max()),
        "mean_window_count": float(counts.mean()),
        "output_count_image_shape": list(counts.shape),
    })

    return counts, point_image, info



def percentile_density_to_uint8(counts, low_percentile=10, high_percentile=90,
                                population="all"):
    """Independent, black-on-white normalization for a single reconstruction.

    10th percentile or below -> 255 (white)
    90th percentile or above -> 0 (black)
    Linear for counts between cutoffs.

    population="all" uses the distribution of ALL window counts, including zeros.
    population="nonzero" uses only counts > 0 to determine the percentile cutoffs,
    but still maps counts <= the lower cutoff (and all zero counts) to white.
    """
    if not (0 <= low_percentile < high_percentile <= 100):
        raise ValueError("Percentiles must satisfy 0 <= low < high <= 100")
    if population not in ("all", "nonzero"):
        raise ValueError("PERCENTILE_POPULATION must be 'all' or 'nonzero'")
    counts = np.asarray(counts)
    if counts.size == 0:
        raise ValueError("Cannot normalize an empty density image")

    selected = counts if population == "all" else counts[counts > 0]
    if selected.size == 0:
        return np.full(counts.shape, 255, dtype=np.uint8), 0.0, 0.0, True

    low, high = np.percentile(selected, [low_percentile, high_percentile])
    low, high = float(low), float(high)
    if high <= low:
        # There is no interval over which to linearly scale grayscale.
        # White is a deliberate no-contrast fallback; report this to the user.
        return np.full(counts.shape, 255, dtype=np.uint8), low, high, True

    darkness = np.clip((counts.astype(np.float32) - low) / (high - low), 0, 1)
    grayscale = 255 - np.rint(255 * darkness).astype(np.uint8)
    return grayscale, low, high, False


## 4. Load the reconstruction, normalize, rotate and flip

All points are used for the density image, just like in the sweep. For a quick check of the chosen orientation, the preview plots only a deterministic subsample of the points. The orange frame outlines the image bounds; points outside are excluded from the density image.

In [ ]:
csv_path = Path(INPUT_CSV).expanduser()
if not csv_path.is_file():
    raise FileNotFoundError(f"Set INPUT_CSV to an existing reconstruction CSV: {csv_path}")

# Only load coordinate columns to reduce memory usage for large graphs.
df = pd.read_csv(csv_path, usecols=lambda col: col in ("x", "y"))
if not {"x", "y"}.issubset(df.columns):
    raise ValueError(f"CSV must contain x and y columns. Found: {list(df.columns)}")
positions = df[["x", "y"]].to_numpy(dtype=np.float64, copy=True)
del df

positions_norm, finite_mask = normalize_pos_95(positions)
del positions
positions_oriented = apply_anchor_transform(
    positions_norm,
    rotation_deg=ROTATION_DEG,
    flip_h=FLIP_HORIZONTAL,
    flip_v=FLIP_VERTICAL,
    x_shift=X_SHIFT,
    y_shift=Y_SHIFT,
    extra_scale=EXTRA_SCALE,
)
del positions_norm

print(f"Loaded: {csv_path}")
print(f"Rows: {len(positions_oriented):,}; finite coordinates: {finite_mask.sum():,}")
print(f"Rotation: {ROTATION_DEG}°; horizontal flip: {FLIP_HORIZONTAL}; "
      f"vertical flip: {FLIP_VERTICAL}")

valid_indices = np.flatnonzero(np.isfinite(positions_oriented).all(axis=1))
rng = np.random.default_rng(SCATTER_SEED)
if len(valid_indices) > PREVIEW_SCATTER_POINTS:
    subset = rng.choice(valid_indices, PREVIEW_SCATTER_POINTS, replace=False)
else:
    subset = valid_indices

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(positions_oriented[subset, 0], positions_oriented[subset, 1],
           s=0.25, c="black", alpha=0.3, linewidths=0, rasterized=True)
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((XLIM[0], YLIM[0]), XLIM[1]-XLIM[0], YLIM[1]-YLIM[0],
                       fill=False, edgecolor="darkorange", linewidth=1.5))
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"Oriented reconstruction (preview of {len(subset):,} points)")
ax.set_xlabel("normalized x")
ax.set_ylabel("normalized y")
plt.show()
plt.close(fig)


## 5. Make and preview the percentile-normalized density image

This calculates a **true square sliding-window count per sampled image center** (not a Gaussian blur). It then maps the 10th and 90th percentile counts to white/black.

**Important:** if P10 and P90 coincide (often when most windows are empty), there is no valid linear contrast range: this notebook produces a white image and prints a warning. Try `PERCENTILE_POPULATION = "nonzero"` to compute thresholds from nonempty windows, or adjust the window width.

In [ ]:
counts, point_image, density_info = local_density_map(
    positions_oriented,
    image_shape=IMAGE_SHAPE,
    window_width_norm=WINDOW_WIDTH_NORM,
    step_fraction=STEP_FRACTION,
    xlim=XLIM,
    ylim=YLIM,
)
density_png, low_count, high_count, degenerate = percentile_density_to_uint8(
    counts,
    low_percentile=LOW_PERCENTILE,
    high_percentile=HIGH_PERCENTILE,
    population=PERCENTILE_POPULATION,
)

density_info.update({
    "low_percentile_count": float(low_count),
    "high_percentile_count": float(high_count),
    "percentile_range_degenerate": bool(degenerate),
    "min_window_count": int(counts.min()),
})
print(f"Raster shape: {IMAGE_SHAPE}; density PNG: {density_png.shape[1]} × {density_png.shape[0]} pixels")
print(f"Counting window: {density_info['window_width_px']} × {density_info['window_height_px']} raster pixels")
print(f"Output stride: {density_info['step_x_px']} × {density_info['step_y_px']} pixels")
print(f"Points within frame: {density_info['points_in_bounds']:,} / {density_info['finite_points_total']:,}")
print(f"P{LOW_PERCENTILE:g} = {low_count:g} counts → white; P{HIGH_PERCENTILE:g} = {high_count:g} counts → black")
if degenerate:
    print("WARNING: Percentile thresholds coincide; the image is white. "
          "Try PERCENTILE_POPULATION='nonzero' or a different window width.")

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(density_png, cmap="gray", vmin=0, vmax=255, interpolation="nearest",
          extent=[XLIM[0], XLIM[1], YLIM[0], YLIM[1]])
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Density preview (not part of exported PNG)")
plt.show()
plt.close(fig)


## 6. Save the **borderless, full-resolution PNG**

The PNG stores the **actual 8-bit grayscale pixel values** without Matplotlib axes, titles, rescaling or interpolation. White is 255, black is 0. `PNG_DPI` sets print-resolution metadata; the true resolution is the number of pixels shown above. A JSON sidecar records your settings for reproducibility.

In [ ]:
if OUTPUT_DIR is None:
    output_dir = csv_path.parent / "density_exports"
else:
    output_dir = Path(OUTPUT_DIR).expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

if OUTPUT_FILENAME is None:
    window_tag = str(WINDOW_WIDTH_NORM).replace(".", "p")
    filename = (f"{csv_path.stem}_density_window{window_tag}_"
                f"P{LOW_PERCENTILE:g}-P{HIGH_PERCENTILE:g}_{PERCENTILE_POPULATION}.png")
else:
    filename = OUTPUT_FILENAME
    if not filename.lower().endswith(".png"):
        filename += ".png"

png_path = output_dir / filename
Image.fromarray(density_png).save(png_path, format="PNG", dpi=(PNG_DPI, PNG_DPI))
print(f"Saved PNG: {png_path.resolve()}")
print(f"PNG dimensions: {density_png.shape[1]} × {density_png.shape[0]} pixels, 8-bit grayscale")

if SAVE_SETTINGS_JSON:
    settings = {
        "source_csv": str(csv_path.resolve()),
        "png_path": str(png_path.resolve()),
        "rotation_deg_ccw": ROTATION_DEG,
        "flip_horizontal_before_rotation": FLIP_HORIZONTAL,
        "flip_vertical_before_rotation": FLIP_VERTICAL,
        "x_shift": X_SHIFT,
        "y_shift": Y_SHIFT,
        "extra_scale": EXTRA_SCALE,
        "position_normalization": "normalize_pos_95",
        "window_width_norm": WINDOW_WIDTH_NORM,
        "image_shape_raster": list(IMAGE_SHAPE),
        "output_shape_pixels": list(density_png.shape),
        "xlim": list(XLIM),
        "ylim": list(YLIM),
        "step_fraction": STEP_FRACTION,
        "low_percentile": LOW_PERCENTILE,
        "high_percentile": HIGH_PERCENTILE,
        "percentile_population": PERCENTILE_POPULATION,
        "png_dpi_metadata": PNG_DPI,
        "image_info": density_info,
    }
    json_path = png_path.with_suffix(".json")
    with open(json_path, "w") as f:
        json.dump(settings, f, indent=2)
    print(f"Saved settings: {json_path.resolve()}")


### Notes

- To change orientation, change `ROTATION_DEG`, `FLIP_HORIZONTAL`, and/or `FLIP_VERTICAL` and rerun cells **4–6**. No alignment is performed because there is only one reconstruction.
- `IMAGE_SHAPE=(4096,4096)` and a one-pixel output stride yield a **4096 × 4096** PNG. For a faster/lighter version, try `(2048,2048)`. To maximize detail, check the printed **output stride** is `1 × 1`; otherwise increase resolution or decrease `STEP_FRACTION`.
- For a **tight crop**, change `XLIM` and `YLIM` together (cropping affects the percentile distribution).
- If nearly everything is white because the 10th and 90th percentiles are both zero, try `PERCENTILE_POPULATION="nonzero"`. This changes **only the percentile thresholds**, not the counts.
- PNG pixel dimensions determine the amount of saved detail; changing `PNG_DPI` alone does **not** create more pixels.
